# **Starter code**

In [266]:
"""
Although we provide a list of permitted packages,
not all of them are necessarily required,
nor is the list guaranteed to be sufficient.
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import json
import joblib

In [267]:
import requests

BASE_URL = "https://usaaio-round1-problem5.billowing-field-1946.workers.dev"
DATASET_TOKEN = "52c963809161730e3fe2da4248a6120f630c0d7815b48e7b36baae864211cfb6"

def download_dataset(filename):
    response = requests.get(
        f"{BASE_URL}/{filename}",
        headers={"Authorization": f"Bearer {DATASET_TOKEN}"},
        timeout=30,
    )
    response.raise_for_status()
    with open(filename, "wb") as f:
        f.write(response.content)
    return filename

train_path = download_dataset("train.json")


In [268]:
with open(train_path, "r") as f:
    data_train = json.load(f)

print(type(data_train))
print(data_train.keys())

# John: Not recommended to convert to numpy array directly.
# Pandas has advantange doing feature engineering.
x_raw = np.array(data_train["x"], dtype=np.float32).reshape(-1, 1)
y_raw = np.array(data_train["y"], dtype=np.float32).reshape(-1, 1)

<class 'dict'>
dict_keys(['x', 'y'])


# Convert data_train from JSON to Pandas DataFrame

In [269]:
# convert data_train into pandas dataframe and store in x_raw, y_raw
import pandas as pd
df = pd.DataFrame(data_train)
print(df.head())

# Take out X, y from df
x_raw = df[["x"]]
y_raw = df["y"]

          x         y
0  0.123620  0.796832
1  1.852143 -0.688261
2  1.195982  0.748218
3  0.795975 -0.860720
4 -0.531944  0.118313


# Test Data Split



In [270]:
# Check the shape of x_raw and y_raw
print(x_raw.shape)
print(y_raw.shape)

# Do the train validation test split
#  - Split test data 20%, train+val 80%
# Look up the cheatsheet https://colab.research.google.com/drive/1L2P3-7s5kpsgizrZ5L9tpYTe1-djOoJT#scrollTo=o1e41qb8U9xW
from sklearn.model_selection import train_test_split
x_train_val, x_test, y_train_val, y_test = train_test_split(x_raw, y_raw, test_size=0.2, random_state=42)

# Check the shape of training validation and test data sets.
print(x_train_val.shape)
print(x_test.shape)
print(y_train_val.shape)
print(y_test.shape)



(8000, 1)
(8000,)
(6400, 1)
(1600, 1)
(6400,)
(1600,)


# Validation Data Split

In [271]:
# Train validation test split
#  - Split validation data 25% of train+val, 75% training data of train+val
# Look up the cheatsheet https://colab.research.google.com/drive/1L2P3-7s5kpsgizrZ5L9tpYTe1-djOoJT#scrollTo=o1e41qb8U9xW
from sklearn.model_selection import train_test_split
x_train, x_val, y_train, y_val = train_test_split(x_train_val, y_train_val, test_size=0.25, random_state=42)

# Check the shape of training validation and test data sets.
print(x_train.shape)
print(x_val.shape)
print(y_train.shape)
print(y_val.shape)

print(x_train)

(4800, 1)
(1600, 1)
(4800,)
(1600,)
             x
7339  0.600851
629  -0.242689
955   0.149416
2151 -0.478715
1316  0.579208
...        ...
6140  0.399124
4170 -0.469309
1761  1.533071
1252 -0.648075
2155 -0.280378

[4800 rows x 1 columns]


# Feature Engineering

In [272]:
# Write a FUNCTION to perform feature engineering
# 1. Add x^2 column to the input matrix
# 2. Add `positive` column which stores if "x" column is positive or not
# 3. Apply standard scaler to x and x^2 column
# 4. Apply one hot encoder to positive column
# 5. IMPORTANT:
#.     During training persist the preprocessor parameter into joblib file.
#.     During validation or testing, load preprocessor parameter from joblib.
#
#. Syntax for scaler and how to persist parameter exists in cheatsheet:
#. https://colab.research.google.com/drive/1L2P3-7s5kpsgizrZ5L9tpYTe1-djOoJT#scrollTo=o1e41qb8U9xW
#
#. Additional Stuff to Memorize
#. A. How to create column in Pandas
#. B. How to drop column in Pandas
#. C. How to convert np array / csr array backto Pandas

import numpy as np
import joblib
from sklearn.preprocessing import StandardScaler, OneHotEncoder

def add_columns(x):
  x["x^2"] = x["x"] ** 2
  x["positive"] = x["x"] > 0

  return x

def apply_scaler(x, column_names, training=True):
  scaler = StandardScaler()
  if training:
    scaler.fit(x[column_names])
    joblib.dump(scaler, "scaler.joblib")
  else:
    scaler = joblib.load("scaler.joblib")

  # The output is np array, convert it back to pandas
  scaled_columns_np = scaler.transform(x[column_names])

  output_column_names = scaler.get_feature_names_out()
  x = x.drop(columns = column_names)

  x[output_column_names] = pd.DataFrame(
      scaled_columns_np,
      columns=output_column_names,
      index=x.index
  )

  return x

def apply_onehot(x, column_names, training=True):
  onehot = OneHotEncoder()
  if training:
    onehot.fit(x[column_names])
    joblib.dump(onehot, "onehot.joblib")
  else:
    onehot = joblib.load("onehot.joblib")

  # The output is csr array, convert it back to pandas
  onehot_columns_np = onehot.transform(x[column_names])

  x = x.drop(columns = column_names)
  output_column_names = onehot.get_feature_names_out()
  x[output_column_names] = pd.DataFrame(
      onehot_columns_np.toarray(),
      columns=output_column_names,
      index = x.index
  )

  return x


def feature_eng(x, training=True):
  x = add_columns(x)
  x = apply_scaler(x, ["x", "x^2"], training)
  x = apply_onehot(x, ["positive"], training)
  return x


x_train = feature_eng(x_train)
print(x_train)
x_val = feature_eng(x_val, training=False)
print(x_val)

             x       x^2  positive_False  positive_True
7339  0.125254 -0.581419             0.0            1.0
629  -0.844619 -0.857005             1.0            0.0
955  -0.393791 -0.890365             0.0            1.0
2151 -1.115993 -0.701691             1.0            0.0
1316  0.100370 -0.604716             0.0            1.0
...        ...       ...             ...            ...
6140 -0.106685 -0.765422             0.0            1.0
4170 -1.105178 -0.709825             1.0            0.0
1761  1.197088  1.233129             0.0            1.0
1252 -1.310718 -0.527620             1.0            0.0
2155 -0.887952 -0.839023             1.0            0.0

[4800 rows x 4 columns]
             x       x^2  positive_False  positive_True
887  -0.893786 -0.836404             1.0            0.0
3732 -0.174498 -0.805194             0.0            1.0
5326 -1.430532 -0.394510             1.0            0.0
667  -1.161111 -0.666016             1.0            0.0
1446  0.673723  0.14903

# Train the Model

In [273]:
# Convert the data you want to input into model into numpy array

x_train_np = x_train.to_numpy()
y_train_np = y_train.to_numpy()
x_val_np = x_val.to_numpy()
y_val_np = y_val.to_numpy()

# Load LinearRegression Model and train the data against it.
# Load model is included in cheatsheet
# https://colab.research.google.com/drive/1L2P3-7s5kpsgizrZ5L9tpYTe1-djOoJT#scrollTo=o1e41qb8U9xW
from sklearn.linear_model import LinearRegression
model = LinearRegression()

# train the model using training data
model.fit(x_train_np, y_train_np)

# write a FUNCTION to calculate the mean_abs_error, and mean_pct_error between
# actual value and predicted values
def mean_abs_error(actual, predicted):
  return np.abs(actual - predicted).mean()

def mean_pct_error(actual, predicted):
  return np.abs((actual - predicted) / actual).mean()

# Get the model output on training data
print("Training Result")
y_train_pred = model.predict(x_train_np)
print(mean_abs_error(y_train, y_train_pred))
print(mean_pct_error(y_train, y_train_pred))


# Get the model output on validation data
print("Validation Result")
y_val_pred = model.predict(x_val_np)
print(mean_abs_error(y_val, y_val_pred))
print(mean_pct_error(y_val, y_val_pred))

# Persist model into a joblib file
joblib.dump(model, "model.joblib")

Training Result
0.5138244965830663
2.087771410756026
Validation Result
0.5058295967340536
2.4504028153895274


['model.joblib']